# Laboratorio visual del ETL — Business Intelligence & Analytics

**Objetivo:** entender, paso a paso, qué hace el módulo `src/etl/procesar_datos.py` del TP: fuentes Excel → limpieza → unión → notas → alertas → tablas analíticas.

**No copiamos ni reescribimos la lógica:** importamos las funciones reales del proyecto y mostramos resultados intermedios. Si cambia el script, este laboratorio utiliza su implementación actual al reiniciar el entorno y volver a clonar.

**Orden:** ejecutar las celdas de arriba hacia abajo. La rama elegida es `matias`; no modifica `main`. Este notebook lee los Excel originales, los cuales **no están guardados en GitHub**. Las transformaciones intermedias se mantienen en memoria y al final se puede ejecutar el ETL completo.

> Las cifras y gráficas dependen de los archivos que cargues. Los ejemplos de vista previa muestran un máximo de algunas filas para no saturar Colab.

## 0. Preparar Colab y clonar el repositorio privado

**Antes de ejecutar:** en Colab abrí el panel de **Secrets** (ícono de llave), creá `GITHUB_TOKEN` con un token personal de GitHub de acceso **solo lectura** a `bi-analytics-tp`, y permití el acceso del notebook al secret. No escribas ni publiques el token en una celda.

El código lo pasa como autenticación temporal a Git. No queda escrito en la URL remota, en el notebook ni en el resultado de las celdas. Si ya tenés el repositorio clonado en `/content/bi-analytics-tp`, lo reutiliza. El token únicamente es necesario para clonar el repositorio privado.

In [ ]:
from pathlib import Path
import os, sys, subprocess, base64

REPO = "https://github.com/matiasjuarez95/bi-analytics-tp.git"
RAMA = "matias"
PROJECT_DIR = Path("/content/bi-analytics-tp")

if not PROJECT_DIR.is_dir():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("Agregá GITHUB_TOKEN en Secrets de Colab y habilitá el acceso del notebook.") from exc
    if not token:
        raise RuntimeError("Falta el secret GITHUB_TOKEN en Colab.")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"
    subprocess.run(["git", "clone", "--quiet", "--branch", RAMA, REPO, str(PROJECT_DIR)],
                   check=True, env=env)
    del token, auth, env
else:
    print("Se reutiliza el repositorio existente; reiniciá el entorno para clonar una versión nueva.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)
from src.etl import procesar_datos as etl
import pandas as pd
from IPython.display import display
print("Repositorio:", PROJECT_DIR)
print("Rama:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

## 1. Leer los Excel directamente desde Google Drive

Los datos de entrada están en esta [carpeta compartida](https://drive.google.com/drive/folders/1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W).

Esta celda abre la autenticación de Google, accede **por el ID de la carpeta** (incluso si está en «Compartidos conmigo») y copia los archivos `.xlsx` al almacenamiento temporal de Colab, conservando las subcarpetas. Así, el ETL puede abrirlos con `pandas.read_excel`. No hace falta descargar ni subir ZIP, ni crear un acceso directo en «Mi unidad».

**Requisito:** autenticarse con una cuenta de Google que tenga permiso de lectura sobre la carpeta. Estos archivos se descargan solo a la sesión temporal de Colab; no se incorporan al repositorio de GitHub. Al reiniciar el entorno, habrá que volver a ejecutar esta celda.

In [ ]:
from pathlib import Path
from io import BytesIO
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import re

# Es el ID de la carpeta de Drive, no una ruta del sistema de archivos.
DRIVE_FOLDER_ID = "1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W"
RAW_DIR = Path("/content/datos_lms")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Autenticación de solo lectura con la cuenta Google que tiene acceso a la carpeta.
auth.authenticate_user()
credenciales, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)
drive = build("drive", "v3", credentials=credenciales, cache_discovery=False)

def listar_hijos(folder_id):
    archivos = []
    pagina = None
    while True:
        respuesta = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000,
            pageToken=pagina,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        archivos.extend(respuesta.get("files", []))
        pagina = respuesta.get("nextPageToken")
        if not pagina:
            break
    return sorted(archivos, key=lambda x: x["name"])

def descargar_carpeta(folder_id, destino):
    cantidad = 0
    for elemento in listar_hijos(folder_id):
        nombre = elemento["name"]
        # No permitimos rutas absolutas ni componentes de directorio en los nombres remotos.
        if nombre in (".", "..") or "/" in nombre or chr(92) in nombre:
            raise ValueError(f"Nombre inseguro en Drive: {nombre}")
        ruta = destino / nombre
        if elemento["mimeType"] == "application/vnd.google-apps.folder":
            ruta.mkdir(parents=True, exist_ok=True)
            cantidad += descargar_carpeta(elemento["id"], ruta)
        elif nombre.lower().endswith(".xlsx"):
            if not ruta.exists():
                solicitud = drive.files().get_media(fileId=elemento["id"], supportsAllDrives=True)
                with ruta.open("wb") as salida:
                    downloader = MediaIoBaseDownload(salida, solicitud)
                    terminado = False
                    while not terminado:
                        _, terminado = downloader.next_chunk()
            cantidad += 1
    return cantidad

cantidad = descargar_carpeta(DRIVE_FOLDER_ID, RAW_DIR)
print(f"Excel disponibles: {cantidad}")
print("Carpeta local para el ETL:", RAW_DIR)

# Comprueba que la carpeta incluya los reportes obligatorios de las 3 materias.
archivos = etl.encontrar_archivos(str(RAW_DIR))
print("Asignaturas detectadas:", ", ".join(archivos.keys()))

## 2. Inventario: ¿qué archivos usa cada asignatura?

`encontrar_archivos()` reconoce nombres con y sin tildes, verifica los tres tipos de reporte y la matrícula, y detiene el proceso si encuentra una fuente obligatoria ausente o ambigua.

In [ ]:
etl.reiniciar_registro()
archivos = etl.encontrar_archivos(str(RAW_DIR))
inventario = []
for materia, fuentes in archivos.items():
    inventario.append({
        "asignatura": materia,
        "matricula": Path(fuentes["matricula"]).name,
        "participacion": Path(fuentes["participacion"]).name,
        "trazabilidad": Path(fuentes["trazabilidad"]).name,
        "archivos_calificaciones": len(fuentes["calificaciones"]),
    })
display(pd.DataFrame(inventario))
print("Total de Excel detectados:", sum(3 + len(v["calificaciones"]) for v in archivos.values()))

## 3. Matrícula: altas, bajas y re-inscripciones

Una persona puede tener varios movimientos. `leer_matricula()` interpreta el archivo; `resolver_matricula()` conserva **una fila vigente por estudiante y materia**, y mantiene todos los movimientos para auditoría.

**Pregunta para explorar:** ¿qué registros quedaron en Baja y cuáles en Aceptada?

In [ ]:
mats, movimientos = [], []
for asig in etl.ASIGNATURAS:
    vig, mov = etl.resolver_matricula(etl.leer_matricula(archivos[asig]["matricula"], asig), asig)
    mats.append(vig)
    movimientos.append(mov)
matricula = pd.concat(mats, ignore_index=True)
matricula_movimientos = pd.concat(movimientos, ignore_index=True)
display(matricula.head(8))
display(pd.crosstab(matricula["asignatura"], matricula["estado_insc"]))
print("Personas con varios movimientos:", matricula_movimientos.loc[
    matricula_movimientos["n_movimientos"] > 1, ["asignatura","id_estudiante"]
].drop_duplicates().shape[0])
display(matricula_movimientos.loc[matricula_movimientos["n_movimientos"] > 1,
    ["asignatura","id_estudiante","estado_insc","fecha_inscripcion","es_vigente"]].head(12))

## 4. Participación: ¿quién ingresó y cuánto participó?

Las fechas, los tiempos de uso y las interacciones se normalizan. Se separan los perfiles de alumnos del personal y se detectan registros con cero interacción.

In [ ]:
participaciones, personal_participacion = [], []
for asig in etl.ASIGNATURAS:
    df = etl.leer_participacion(archivos[asig]["participacion"], asig)
    alumnos, personal = etl.procesar_participacion(df, asig)
    participaciones.append(alumnos)
    personal_participacion.append(personal)
participacion = pd.concat(participaciones, ignore_index=True)
display(participacion[["asignatura","id_estudiante","interacciones",
                        "tiempo_uso_min","nunca_ingreso"]].head(10))
display(participacion.groupby("asignatura", as_index=False).agg(
    estudiantes=("id_estudiante","nunique"),
    interacciones=("interacciones","sum"),
    nunca_ingresaron=("nunca_ingreso","sum"),
    tiempo_medio_min=("tiempo_uso_min","mean")
).round(1))

## 5. Trazabilidad: eventos, duplicados y tipos de interacción

A diferencia de la participación, la trazabilidad contiene **muchos eventos por estudiante**. El ETL conserva eventos idénticos y los marca como duplicados *potenciales*; no asume que sean errores.

In [ ]:
trazas, personal_trazabilidad = [], []
for asig in etl.ASIGNATURAS:
    tr, n_original = etl.leer_trazabilidad(archivos[asig]["trazabilidad"], asig)
    alumnos, personal = etl.separar_alumnos_personal(tr, "trazabilidad", asig)
    trazas.append(alumnos)
    personal_trazabilidad.append(personal)
trazabilidad = pd.concat(trazas, ignore_index=True)
display(trazabilidad[["asignatura","id_estudiante","fecha","accion",
                       "tipo_interaccion","duplicado_potencial"]].head(10))
display(trazabilidad.groupby(["asignatura","tipo_interaccion"]).size().unstack(fill_value=0))
print("Eventos potencialmente repetidos:", int(trazabilidad["duplicado_potencial"].sum()))

In [ ]:
import plotly.express as px
tipo_por_materia = (trazabilidad.groupby(["asignatura","tipo_interaccion"])
                    .size().reset_index(name="eventos"))
fig = px.bar(tipo_por_materia, x="asignatura", y="eventos",
             color="tipo_interaccion", barmode="stack",
             title="Eventos del LMS por tipo y asignatura")
fig.show()

## 6. Calificaciones: formato largo y recuperatorios

Cada archivo Excel representa un instrumento. El ETL lee los archivos, extrae nota y máximo, y genera **una fila por estudiante e instrumento**.

Los recuperatorios tienen una lógica particular: el código señala cuáles notas se usan y cuáles no, pero no elimina los registros históricos.

In [ ]:
calificaciones_list = []
for asig in etl.ASIGNATURAS:
    for ruta in archivos[asig]["calificaciones"]:
        calificaciones_list.append(etl.leer_calificaciones(ruta, asig))
calificaciones_crudas = pd.concat(calificaciones_list, ignore_index=True)
calificaciones = etl.aplicar_recuperatorios(calificaciones_crudas, modo="docente")
display(calificaciones[["id_estudiante","asignatura","instrumento","puntos","maximo",
                         "nota_100","es_recuperatorio","usa_en_nota_final"]].head(12))
display(calificaciones.groupby(["asignatura","tipo_instrumento"]).agg(
    registros=("id_estudiante","size"),
    notas_utilizadas=("usa_en_nota_final","sum")
).reset_index())
display(calificaciones.loc[calificaciones["es_recuperatorio"],
     ["id_estudiante","instrumento","nota_100","usa_en_nota_final"]].head(15))

## 7. Universo de alumnos: integración sin duplicar estudiantes

El universo es la unión de quienes aparecen en matrícula, participación, trazabilidad o calificaciones, usando `(id_estudiante, asignatura)` como clave.

Un alumno puede estar en varias materias: lo contamos una vez **por materia**, no una vez por evento.

In [ ]:
universo = pd.concat([
    matricula[["id_estudiante","asignatura"]],
    participacion[["id_estudiante","asignatura"]],
    calificaciones[["id_estudiante","asignatura"]],
    trazabilidad[["id_estudiante","asignatura"]],
], ignore_index=True).drop_duplicates().reset_index(drop=True)
print("Filas del universo:", len(universo))
print("Claves duplicadas:", universo.duplicated(["id_estudiante","asignatura"]).sum())
display(universo.groupby("asignatura").size().rename("alumnos").reset_index())

## 8. Notas finales: real vs. estimado

**Álgebra** acumula puntos reales, pero también calcula una proyección sobre los instrumentos disponibles. Las proyecciones **no determinan el estado definitivo**.

**Probabilidad y Algoritmos** usan promedios; el ETL distingue promedio observado y promedio considerando cero para exámenes presentes que no se rindieron.

El `estado_provisorio` y el `estado_definitivo` no significan lo mismo.

In [ ]:
notas_finales = etl.calcular_notas_finales(calificaciones, universo, cursada_cerrada=False)
columnas = ["asignatura","id_estudiante","instrumentos_rendidos","nota_a_la_fecha",
            "estado_provisorio","estado_definitivo"]
display(notas_finales[columnas].head(12))
algebra = notas_finales[notas_finales["asignatura"] == "Álgebra"]
display(algebra[["id_estudiante","puntos_acumulados","puntos_maximos_observados",
                 "proyeccion_120_observados","estado_provisorio","estado_definitivo"]].head(12))
display(pd.crosstab(notas_finales["asignatura"], notas_finales["estado_provisorio"]))

## 9. Tabla integrada: situación de cada estudiante

El ETL agrega por estudiante-materia los eventos, días activos, primer y último evento; luego combina todo con matrícula, participación y notas.

**Regla clave:** `situacion_estudiante` debe tener exactamente una fila por `(id_estudiante, asignatura)`.

In [ ]:
eventos = trazabilidad.groupby(["id_estudiante","asignatura"]).agg(
    eventos_trazabilidad=("fecha","size"),
    eventos_repetidos_potenciales=("duplicado_potencial","sum"),
    dias_activos=("fecha_dia","nunique"),
    primer_evento=("fecha","min"),
    ultimo_evento=("fecha","max"),
).reset_index()
situacion = etl.construir_situacion(universo, matricula, participacion, eventos, notas_finales)
print("Filas del universo:",len(universo),"| Filas del modelo:",len(situacion))
assert not situacion.duplicated(["id_estudiante","asignatura"]).any()
display(situacion[["asignatura","id_estudiante","matriculado",
                   "tiene_actividad","situacion_matricula","estado_provisorio","semaforo"]].head(15))
display(pd.crosstab(situacion["asignatura"], situacion["situacion_matricula"]))

## 10. Calidad: ¿qué se descartó y qué quedó marcado?

El registro de calidad `LOG` guarda decisiones, afectados y balances. `DESCARTADOS` conserva filas descartadas. Mirá las causas antes de confiar en una métrica.

In [ ]:
calidad = pd.DataFrame(etl.LOG)
descartados = (pd.concat(etl.DESCARTADOS, ignore_index=True) if etl.DESCARTADOS
              else pd.DataFrame(columns=["fuente","asignatura","motivo"]))
print("Registros de calidad:",len(calidad),"| Filas descartadas:",len(descartados))
display(calidad[["fuente","asignatura","problema","registros_afectados","decision"]].head(25))
display(descartados.head(10))

## 11. Validar el modelo

Esta celda ejecuta las comprobaciones que ya implementó Lara: claves únicas, rangos de notas, semáforos, matrícula e integridad de la integración. Una validación `ok=False` requiere análisis antes de avanzar a los KPI.

In [ ]:
personal = pd.concat(personal_participacion + personal_trazabilidad, ignore_index=True)
validaciones = etl.validar_modelo(
    situacion, notas_finales, calificaciones, matricula, universo, personal,
    cursada_cerrada=False
)
display(validaciones)
assert validaciones["ok"].all(), "Existen validaciones fallidas: revisar antes de continuar."

## 12. Ejecutar exactamente el ETL completo (punta a punta)

Hasta acá inspeccionamos cada paso con DataFrames en memoria. Ahora llamamos al orquestador oficial `etl.ejecutar()`, que reproduce el pipeline de producción y escribe las tablas procesadas.

**Nota:** puede generar CSV (y opcionalmente Parquet). Eso no convierte ni modifica las **fuentes originales Excel**.

In [ ]:
SALIDA = Path("/content/etl_resultados")
salidas = etl.ejecutar(str(RAW_DIR), str(SALIDA))
resumen_salidas = pd.DataFrame([{"tabla":nombre, "filas":len(df), "columnas":len(df.columns)}
                                for nombre,df in salidas.items()])
display(resumen_salidas)
print("Archivos generados:",sorted(p.name for p in SALIDA.iterdir()))

## 13. Pruebas automatizadas

Además de la exploración visual, ejecutá los tests existentes con datos sintéticos. Estos tests verifican reglas y casos límite **sin depender de los Excel**.

In [ ]:
resultado = subprocess.run([sys.executable,"-m","pytest","tests/test_etl_6_1.py","-q"],
                            cwd=str(PROJECT_DIR), text=True)
if resultado.returncode:
    raise RuntimeError(f"Fallaron las pruebas (código {resultado.returncode}).")

## Qué sigue

Este notebook es un **laboratorio de lectura y exploración**, no otro ETL distinto. La lógica reside únicamente en `src/etl/procesar_datos.py`.

Para futuras correcciones, modificá los scripts en tu rama, hacé `push` y luego **reiniciá el runtime de Colab** para clonar la versión nueva. El notebook no modifica las fuentes ni hace `push` a GitHub.

**Atención:** la columna `semaforo_definitivo` del ETL actual devuelve un booleano y tiene una corrección pendiente en nuestra rama. No confundas esa columna con el semáforo de colores `semaforo`.